# Arquitecturas Multi-Agente

En el notebook anterior construiste el primitivo fundamental: el **agentic loop**.
Ahora lo usarás como bloque de construcción para sistemas más complejos.

## El problema del agente monolítico

Un solo agente con demasiadas responsabilidades se vuelve:
- **Difícil de depurar** — ¿en qué paso falló?
- **Poco confiable** — un prompt largo confunde al LLM
- **No reutilizable** — no puedes extraer solo la parte de "investigación"

## La solución: especialización + composición

| Patrón | Cuándo usarlo | Implementación |
|---|---|---|
| **Orquestador LLM** | El orden es dinámico | Sub-agentes como tools |
| **Sequential** | Pipeline fijo, orden garantizado | Llamadas en secuencia |
| **Parallel** | Tareas independientes | `ThreadPoolExecutor` |
| **Loop** | Refinamiento iterativo | `while not approved` |

---
## Sección 1: Setup

In [ ]:
%pip install openai httpx --quiet

In [2]:
import os
import json
import time
import httpx
import urllib.parse
import concurrent.futures
from openai import OpenAI
from dotenv import load_dotenv
from ddgs import DDGS

# ════════════════════════════════════════════════════════════
#  CONFIGURACIÓN — elige tu backend aquí
# ════════════════════════════════════════════════════════════
load_dotenv()

BACKEND = "nvidia"   # opciones: "ollama" | "anthropic" | "openai" | "groq" | "nvidia"

CONFIGS = {
    "ollama": {
        "base_url": "http://localhost:11434/v1",
        "api_key":  "ollama",
        "model":    "qwen2.5:7b",
    },
    "anthropic": {
        "base_url": "https://api.anthropic.com/v1",
        "api_key":  os.getenv("ANTHROPIC_API_KEY", ""),
        "model":    "claude-3-5-haiku-20241022",
    },
    "openai": {
        "base_url": None,
        "api_key":  os.getenv("OPENAI_API_KEY", ""),
        "model":    "gpt-4o-mini",
    },
    "groq": {
        "base_url": "https://api.groq.com/openai/v1",
        "api_key":  os.getenv("GROQ_API_KEY", ""),
        "model":    "llama-3.3-70b-versatile",
    },
    # NVIDIA NIM (catálogo de 50+ modelos, tier gratuito con 1,000 créditos)
    # Requiere: NVIDIA_API_KEY en entorno  →  https://build.nvidia.com
    # El API key empieza con nvapi-
    # Modelos con tool use: meta/llama-3.3-70b-instruct, mistralai/mistral-large-latest
    "nvidia": {
        "base_url": "https://integrate.api.nvidia.com/v1",
        "api_key":  os.getenv("NVIDIA_API_KEY", ""),
        "model":    "meta/llama-3.3-70b-instruct",  # o cualquier otro del catálogo
    },
}

cfg = CONFIGS[BACKEND]
client_kwargs = {"api_key": cfg["api_key"]}
if cfg["base_url"]:
    client_kwargs["base_url"] = cfg["base_url"]

client = OpenAI(**client_kwargs)
MODEL  = cfg["model"]

print(f"✅ Backend: {BACKEND.upper()} | Modelo: {MODEL}")

✅ Backend: NVIDIA | Modelo: meta/llama-3.3-70b-instruct


In [3]:
# ── Herramienta compartida ────────────────────────────────────────────────────

def web_search(query: str) -> str:
    """Busca información actualizada en la web."""
    try:
        results = DDGS().text(query, max_results=3)
        if not results:
            return "No se encontraron resultados."
        output = []
        for r in results:
            output.append(f"**{r['title']}**\n{r['body']}")
        return "\n\n".join(output)
    except Exception as e:
        return f"Error al buscar: {e}"


SEARCH_REGISTRY = {"web_search": web_search}

SEARCH_TOOL = [{
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Busca información actualizada en la web.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Consulta de búsqueda."}
            },
            "required": ["query"]
        }
    }
}]


# ── Primitivo reutilizable: agentic loop ──────────────────────────────────────

def run_agent(
    user_message: str,
    system: str,
    tools: list = None,
    tool_registry: dict = None,
    verbose: bool = True,
    label: str = "Agente",
) -> str:
    """Ejecuta el agentic loop. Retorna el texto final de respuesta."""
    tools         = tools         or []
    tool_registry = tool_registry or {}

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": user_message},
    ]

    if verbose:
        preview = user_message[:100] + "..." if len(user_message) > 100 else user_message
        print(f"  [{label}] ← {preview}")

    for _ in range(10):
        kwargs = {"model": MODEL, "messages": messages}
        if tools:
            kwargs["tools"]       = tools
            kwargs["tool_choice"] = "auto"

        response      = client.chat.completions.create(**kwargs)
        choice        = response.choices[0]
        finish_reason = choice.finish_reason

        if finish_reason == "stop":
            text = choice.message.content or ""
            if verbose:
                preview = text[:150] + "..." if len(text) > 150 else text
                print(f"  [{label}] → {preview}")
            return text

        if finish_reason == "tool_calls":
            messages.append(choice.message)
            for tc in choice.message.tool_calls:
                args   = json.loads(tc.function.arguments)
                fn     = tool_registry.get(tc.function.name)
                result = fn(**args) if fn else f"Tool '{tc.function.name}' no encontrada."
                if verbose:
                    print(f"  [{label}] 🔧 {tc.function.name}({args})")
                messages.append({
                    "role": "tool", "tool_call_id": tc.id, "content": result
                })
        else:
            break

    return "[Límite de iteraciones alcanzado]"


print("✅ Primitivos listos.")

✅ Primitivos listos.


---
## Sección 2: Orquestador LLM — Decisión Dinámica

Un agente coordinador que usa a otros agentes como herramientas.
El LLM decide en tiempo de ejecución qué sub-agente llamar y en qué orden.

```
Usuario
  └─► Coordinador (LLM)
         ├─► research_agent  (tool call → run_agent interno)
         └─► summarizer_agent (tool call → run_agent interno)
```

In [4]:
# ── Sub-agentes como funciones Python ────────────────────────────────────────

def research_agent(topic: str) -> str:
    return run_agent(
        user_message=f"Investiga: {topic}",
        system=(
            "Eres un agente de investigación. Usa web_search para encontrar "
            "2-3 datos relevantes sobre el tema. Presenta hallazgos con fuentes."
        ),
        tools=SEARCH_TOOL,
        tool_registry=SEARCH_REGISTRY,
        label="ResearchAgent",
    )


def summarizer_agent(research_findings: str) -> str:
    return run_agent(
        user_message=f"Resume estos hallazgos:\n\n{research_findings}",
        system=(
            "Eres un agente de síntesis. Crea un resumen en 3-5 puntos clave "
            "con viñetas. No uses herramientas."
        ),
        label="SummarizerAgent",
    )


# ── Tools del coordinador: los sub-agentes expuestos como herramientas ────────

ORCHESTRATOR_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "research_agent",
            "description": "Busca información actualizada sobre un tema y devuelve hallazgos con fuentes.",
            "parameters": {
                "type": "object",
                "properties": {"topic": {"type": "string"}},
                "required": ["topic"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "summarizer_agent",
            "description": "Crea un resumen conciso a partir de hallazgos de investigación.",
            "parameters": {
                "type": "object",
                "properties": {"research_findings": {"type": "string"}},
                "required": ["research_findings"]
            }
        }
    },
]

ORCHESTRATOR_REGISTRY = {
    "research_agent":   research_agent,
    "summarizer_agent": summarizer_agent,
}

print("✅ Sub-agentes y tools del orquestador listos.")

✅ Sub-agentes y tools del orquestador listos.


In [5]:
print("=" * 60)
print("ORQUESTADOR LLM: Research & Summarization")
print("=" * 60)

run_agent(
    user_message="¿Cuáles son los últimos avances en computación cuántica y qué implican para la IA?",
    system=(
        "Eres un coordinador de investigación. Para responder:\n"
        "1. Llama a `research_agent` para obtener información actualizada.\n"
        "2. Llama a `summarizer_agent` con los hallazgos.\n"
        "3. Presenta el resumen final al usuario."
    ),
    tools=ORCHESTRATOR_TOOLS,
    tool_registry=ORCHESTRATOR_REGISTRY,
    label="Coordinador",
)

ORQUESTADOR LLM: Research & Summarization
  [Coordinador] ← ¿Cuáles son los últimos avances en computación cuántica y qué implican para la IA?
  [ResearchAgent] ← Investiga: computación cuántica y su impacto en la IA
  [ResearchAgent] 🔧 web_search({'query': 'computación cuántica y su impacto en la IA'})
  [ResearchAgent] → El impacto de la computación cuántica en la IA es significativo, ya que puede mejorar el poder de cómputo, la eficiencia de los algoritmos y las capac...
  [Coordinador] 🔧 research_agent({'topic': 'computación cuántica y su impacto en la IA'})
  [SummarizerAgent] ← Resume estos hallazgos:

El impacto de la computação quântica na IA é significativo, pois pode melho...
  [SummarizerAgent] → Aqui estão os pontos principais sobre o impacto da computação quântica na Inteligência Artificial (IA):

*   **Melhoria do poder de cómputo e eficiênc...
  [Coordinador] 🔧 summarizer_agent({'research_findings': 'El impacto de la computação quântica na IA é significativo, pois pode m

'El impacto de la computación cuántica en la IA es significativo, ya que puede mejorar el poder de cómputo, la eficiencia de los algoritmos y las capacidades generales de resolución de problemas. La integración de la computación cuántica en la IA puede abrir nuevas fronteras en el diseño de nuevos materiais, el descubrimiento de fármacos y la ingeniería de baterías de alta eficiencia. Sin embargo, también existen desventajas, como la complejidad y el costo de implementar la computación cuántica en la IA.\n\nA continuación, se presentan los puntos principales sobre el impacto de la computación cuántica en la Inteligencia Artificial (IA):\n\n*   **Mejoria del poder de cómputo y eficiencia de los algoritmos**: La computación cuántica puede aumentar significativamente la capacidad de procesamiento de la IA, permitiendo que ella resuelva problemas complejos de forma más eficiente.\n*   **Nuevas fronteras en proyectos y descubrimientos**: La integración de la computación cuántica en la IA pu

---
## Sección 3: Pipeline Secuencial — Orden Garantizado

Cuando el orden **importa** y no quieres depender del LLM para respetarlo,
defines el pipeline en Python. El estado se pasa explícitamente entre pasos.

```
topic → [OutlineAgent] → outline
                  └──► [WriterAgent] → draft
                               └──► [EditorAgent] → resultado final
```

In [6]:
def run_sequential_pipeline(topic: str, verbose: bool = True) -> dict:
    """Pipeline secuencial: Outline → Writer → Editor."""
    state = {}

    if verbose:
        print("=" * 60)
        print(f"PIPELINE SECUENCIAL: Blog sobre '{topic}'")
        print("=" * 60)

    # Paso 1: Outline
    state["outline"] = run_agent(
        user_message=f"Crea un outline de blog sobre: {topic}",
        system=(
            "Crea un outline con título llamativo, hook de introducción, "
            "3-5 secciones con 2-3 puntos cada una, y conclusión. "
            "Solo el outline, sin texto adicional."
        ),
        verbose=verbose, label="OutlineAgent",
    )

    # Paso 2: Writer — recibe el output del paso anterior
    state["draft"] = run_agent(
        user_message=f"Escribe un blog basado en este outline:\n\n{state['outline']}",
        system="Sigue el outline estrictamente. Blog de 200-300 palabras, tono atractivo.",
        verbose=verbose, label="WriterAgent",
    )

    # Paso 3: Editor — recibe el draft
    state["final"] = run_agent(
        user_message=f"Edita y mejora este borrador:\n\n{state['draft']}",
        system=(
            "Corrige errores gramaticales, mejora flujo y claridad. "
            "Mantén la longitud similar. Solo devuelve el texto editado."
        ),
        verbose=verbose, label="EditorAgent",
    )

    return state


state = run_sequential_pipeline(
    "Los beneficios de los sistemas multi-agente para desarrolladores de software"
)

print("\n" + "=" * 60)
print("RESULTADO FINAL:")
print("=" * 60)
print(state["final"])

PIPELINE SECUENCIAL: Blog sobre 'Los beneficios de los sistemas multi-agente para desarrolladores de software'
  [OutlineAgent] ← Crea un outline de blog sobre: Los beneficios de los sistemas multi-agente para desarrolladores de s...
  [OutlineAgent] → # Los beneficios de los sistemas multi-agente para desarrolladores de software
## Introducción
* Definición de sistemas multi-agente
* Importancia de ...
  [WriterAgent] ← Escribe un blog basado en este outline:

# Los beneficios de los sistemas multi-agente para desarrol...
  [WriterAgent] → **Los beneficios de los sistemas multi-agente para desarrolladores de software**

## Introducción

Los sistemas multi-agente son un enfoque innovador ...
  [EditorAgent] ← Edita y mejora este borrador:

**Los beneficios de los sistemas multi-agente para desarrolladores de...
  [EditorAgent] → **Los beneficios de los sistemas multi-agente para desarrolladores de software**

## Introducción
Los sistemas multi-agente representan un enfoque inn...

RESU

---
## Sección 4: Ejecución Paralela — Independiente y Rápido

Cuando las tareas son **independientes**, ejecutarlas en secuencia
desperdicia tiempo. `ThreadPoolExecutor` las lanza simultáneamente.

```
Input
  ├─► [TechResearcher]    ─┐
  ├─► [HealthResearcher]  ─┼─► [Aggregator] → Reporte final
  └─► [FinanceResearcher] ─┘
       (los 3 corren en paralelo)
```

In [7]:
def make_domain_researcher(domain_label: str, instructions: str):
    """Factory: devuelve una función-agente especializada en un dominio."""
    def researcher(prompt: str) -> str:
        return run_agent(
            user_message=prompt,
            system=instructions,
            tools=SEARCH_TOOL,
            tool_registry=SEARCH_REGISTRY,
            label=domain_label,
        )
    return researcher


researchers = {
    "tech": make_domain_researcher(
        "TechResearcher",
        "Investiga las últimas tendencias en IA/ML. "
        "3 desarrollos clave, empresas involucradas e impacto. Máx 100 palabras."
    ),
    "health": make_domain_researcher(
        "HealthResearcher",
        "Investiga avances médicos recientes. "
        "3 avances, aplicaciones prácticas y plazos. Máx 100 palabras."
    ),
    "finance": make_domain_researcher(
        "FinanceResearcher",
        "Investiga tendencias actuales en fintech. "
        "3 tendencias, implicaciones de mercado y perspectivas. Máx 100 palabras."
    ),
}

print("✅ Investigadores especializados listos.")

✅ Investigadores especializados listos.


In [8]:
print("=" * 60)
print("SISTEMA PARALELO: Briefing Ejecutivo Diario")
print("=" * 60)

query = "Dame el briefing del día sobre tu área"
t0    = time.time()

# Lanzar los 3 investigadores en paralelo
with concurrent.futures.ThreadPoolExecutor(max_workers=3) as pool:
    futures = {domain: pool.submit(fn, query) for domain, fn in researchers.items()}
    results = {domain: f.result() for domain, f in futures.items()}

print(f"\n⏱️  Investigación paralela completada en {time.time()-t0:.1f}s")

# Agregar resultados
combined = (
    f"**Tecnología:**\n{results['tech']}\n\n"
    f"**Salud:**\n{results['health']}\n\n"
    f"**Finanzas:**\n{results['finance']}"
)

final_report = run_agent(
    user_message=f"Sintetiza estos tres reportes en un resumen ejecutivo:\n\n{combined}",
    system=(
        "Combina los tres reportes en ~200 palabras. "
        "Destaca temas comunes, conexiones y los 3 takeaways más importantes."
    ),
    label="Aggregator",
)

print("\n" + "=" * 60)
print("REPORTE EJECUTIVO:")
print("=" * 60)
print(final_report)

SISTEMA PARALELO: Briefing Ejecutivo Diario
  [TechResearcher] ← Dame el briefing del día sobre tu área
  [HealthResearcher] ← Dame el briefing del día sobre tu área
  [FinanceResearcher] ← Dame el briefing del día sobre tu área
  [FinanceResearcher] 🔧 web_search({'query': 'tendencias fintech actual'})
  [FinanceResearcher] 🔧 web_search({'query': 'tendencias fintech actual'})  [TechResearcher] 🔧 web_search({'query': 'últimas tendencias en IA/ML'})

  [FinanceResearcher] 🔧 web_search({'query': 'tendencias fintech actual'})
  [FinanceResearcher] 🔧 web_search({'query': 'tendencias fintech actual'})
  [TechResearcher] 🔧 web_search({'query': 'noticias de hoy sobre inteligencia artificial'})
  [FinanceResearcher] 🔧 web_search({'query': 'tendencias fintech actual'})
  [TechResearcher] 🔧 web_search({'query': 'desarrollos clave en inteligencia artificial'})
  [TechResearcher] 🔧 web_search({'query': 'empresas involucradas en inteligencia artificial'})
  [HealthResearcher] 🔧 web_search({'query': 

---
## Sección 5: Loop de Refinamiento — Escritor ⇆ Crítico

Para tareas que requieren **mejora iterativa**. La condición de salida
se verifica en Python — no dependemos del LLM para decidir si continuar.

```
prompt → [InitialWriter] → historia v1
              ↓
    ┌──── LOOP ────────────────┐
    │  [Critic] → feedback     │
    │      ↓                   │
    │  "APPROVED"?             │
    │    Sí → SALIR            │
    │    No → [Refiner] → v+1  │
    └──────────────────────────┘
```

In [9]:
def run_refinement_loop(
    initial_prompt: str,
    max_iterations: int = 3,
    verbose: bool = True,
) -> dict:
    """Loop de refinamiento: Writer ⇆ Critic hasta aprobación o max_iterations."""

    if verbose:
        print("=" * 60)
        print("LOOP DE REFINAMIENTO: Escritor ⇆ Crítico")
        print("=" * 60)

    # Primera versión
    story = run_agent(
        user_message=initial_prompt,
        system="Escribe el primer borrador de una historia corta (100-150 palabras). Solo el texto.",
        verbose=verbose, label="InitialWriter",
    )

    critique = ""
    approved = False

    for i in range(max_iterations):
        if verbose:
            print(f"\n{'─'*60}")
            print(f"  Iteración {i+1}/{max_iterations}")

        # Crítico evalúa
        critique = run_agent(
            user_message=f"Evalúa esta historia:\n\n{story}",
            system=(
                "Eres un crítico constructivo. Evalúa trama, personajes y ritmo.\n"
                "- Si es sólida y completa, responde EXACTAMENTE: APPROVED\n"
                "- Si no, da 2-3 sugerencias específicas y accionables."
            ),
            verbose=verbose, label="Critic",
        )

        # Condición de salida — verificada en Python, no en el LLM
        if "APPROVED" in critique.upper():
            approved = True
            if verbose:
                print(f"  ✅ Historia aprobada en iteración {i+1}.")
            break

        # Refinador incorpora el feedback
        story = run_agent(
            user_message=(
                f"Historia actual:\n{story}\n\n"
                f"Crítica:\n{critique}\n\n"
                "Reescribe incorporando todo el feedback."
            ),
            system="Eres un escritor que refina historias. Incorpora TODO el feedback. Solo el texto.",
            verbose=verbose, label="Refiner",
        )

    return {"story": story, "critique": critique, "iterations": i + 1, "approved": approved}


result = run_refinement_loop(
    initial_prompt="Escribe una historia sobre un guardián de faro que descubre un mapa que brilla en la oscuridad.",
    max_iterations=3,
)

print("\n" + "=" * 60)
print("HISTORIA FINAL:")
print("=" * 60)
print(result["story"])
print(f"\n📊 {result['iterations']} iteraciones | Aprobada: {result['approved']}")

LOOP DE REFINAMIENTO: Escritor ⇆ Crítico
  [InitialWriter] ← Escribe una historia sobre un guardián de faro que descubre un mapa que brilla en la oscuridad.
  [InitialWriter] → En la cima del faro, el guardián, un hombre solitario y experimentado, realizaba su rutina nocturna. Mientras revisaba los instrumentos, notó un papel...

────────────────────────────────────────────────────────────
  Iteración 1/3
  [Critic] ← Evalúa esta historia:

En la cima del faro, el guardián, un hombre solitario y experimentado, realiz...
  [Critic] → APPROVED
  ✅ Historia aprobada en iteración 1.

HISTORIA FINAL:
En la cima del faro, el guardián, un hombre solitario y experimentado, realizaba su rutina nocturna. Mientras revisaba los instrumentos, notó un papel antiguo en un rincón olvidado. Al desdoblarlo, se sorprendió al ver un mapa que brillaba suavemente en la oscuridad. Las líneas y símbolos parecían danzar en la penumbra, iluminando un camino desconocido. El guardián se sintió atraído por el mist

---
## Sección 6: Tu Turno

Elige un patrón y modifícalo para tu propio caso de uso:

**Sequential:** Pipeline para analizar un paper
- `[Extractor]` → puntos clave → `[Traductor]` → `[Simplificador]` para no técnicos

**Parallel:** Comparar 3 modelos de IA en paralelo

**Loop:** Generador de preguntas de examen con auto-revisión de dificultad

In [ ]:
# 👇 Tu implementación
pass

---
## ✅ Resumen

Implementaste los cuatro patrones de arquitectura multi-agente **desde cero**.

| Patrón | Python | Cuándo |
|---|---|---|
| Orquestador LLM | sub-agentes como tools en `run_agent` | El modelo decide el flujo |
| Sequential | llamadas en secuencia + state dict | Pasos que dependen entre sí |
| Parallel | `ThreadPoolExecutor` | Tareas independientes |
| Loop | `while not approved` | Refinamiento iterativo |

Los frameworks como LangGraph o ADK implementan exactamente estos mismos patrones
con abstracciones adicionales. Ahora que entiendes el núcleo, cualquier framework
te resultará completamente transparente.